# Leakage-controlled KPCA-DA candidate run

This notebook delegates training to the canonical matched runner; it does not contain a second copy of the preprocessing or model-selection pipeline. The pair uses the corrected official GSE45498 reconstruction, the current source-only preprocessing and validation protocol, and identical split/classifier settings for both arms. The baseline is the zero-optimization KPCA map control. The paper-style raw DeepInsight baseline is evaluated separately by the queued seed-51 comparison.

The launch cell schedules one development pair (seed 52) after the deterministic-filter seed-46 comparison completes. It returns immediately. Do not use this same held-out split repeatedly to select policy; freeze any candidate before one-time scoring.


In [ ]:
from pathlib import Path
import subprocess

repo_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "audit" / "launch_detached.py").is_file()), None)
if repo_root is None:
    raise RuntimeError("Open this notebook from the repository or one of its subdirectories.")
python = repo_root / ".venv" / "bin" / "python"
if not python.is_file():
    raise RuntimeError(f"Repository environment is missing: {python}")

job = "full_tcga_seed52_corrected_pair"
audit = repo_root / "audit"
job_marker = audit / f"{job}.queued"
existing = [audit / f"{job}.{suffix}" for suffix in ("queued", "log", "status", "done")]
existing = [path for path in existing if path.exists()]
if existing:
    print(f"{job} is already queued or has run; inspect {existing[0].relative_to(repo_root)} instead of launching a duplicate.")
else:
    job_marker.write_text("Queued by Scripts/test_kPCADA.ipynb\n", encoding="utf-8")
    command = [
        str(python), str(audit / "launch_detached.py"), job, "--",
        str(python), str(audit / "run_full_tcga_candidate_pair.py"),
        "--after-job", "exact_variance_filter_seed46_pair_final",
        "--seed", "52",
        "--baseline-arm", "baseline",
        "--feature-scope", "all",
        "--comparison", "audit/full_tcga_seed52_corrected_pair_comparison.json",
    ]
    try:
        subprocess.run(command, cwd=repo_root, check=True)
    except Exception:
        job_marker.unlink(missing_ok=True)
        raise
    print(f"Queued {job}; the detached waiter will record completion in audit/{job}.done")


## Reading the result

The comparison artifact is written to `audit/full_tcga_seed52_corrected_pair_comparison.json` after both arms finish and pass paired verification. Review balanced accuracy, macro-F1, Tumor and Normal sensitivity, and the patient-cluster interval together. A single split is development evidence, not confirmation of generalization. The detached worker log is `audit/full_tcga_seed52_corrected_pair.log`; the completion waiter records `audit/full_tcga_seed52_corrected_pair.done`.
